# Pandas ↔ Scikit-Learn Pipelines

## Definition
A **scikit-learn Pipeline** chains preprocessing steps and a model into a single object that:
- Prevents data leakage (fit only on train data)
- Makes code reproducible and clean
- Allows clean cross-validation
- Can be pickled and deployed as a single artifact

### Pipeline Architecture
```
pandas DataFrame
       ↓
ColumnTransformer
  ├── numeric columns → [Imputer → Scaler]
  └── categorical cols → [Imputer → Encoder]
       ↓
  Transformed NumPy array
       ↓
  ML Model (LogReg, RandomForest, etc.)
       ↓
  Predictions
```

### Key Classes
| Class | Module | Purpose |
|-------|--------|---------|
| `Pipeline` | `sklearn.pipeline` | Chain steps sequentially |
| `ColumnTransformer` | `sklearn.compose` | Apply different transforms to different cols |
| `SimpleImputer` | `sklearn.impute` | Fill missing values |
| `StandardScaler` | `sklearn.preprocessing` | Z-score normalization |
| `OneHotEncoder` | `sklearn.preprocessing` | Categorical encoding |
| `set_output('pandas')` | sklearn >= 1.2 | Keep DataFrame output through pipeline |

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score

rng = np.random.default_rng(42)
n = 600
df = pd.DataFrame({
    'age':        rng.integers(18, 65, n).astype(float),
    'income':     rng.normal(60000, 20000, n).clip(0),
    'credit_score': rng.normal(700, 80, n).clip(300, 850),
    'loan_amount':  rng.normal(25000, 10000, n).clip(1000),
    'employment': rng.choice(['Employed', 'Self-Employed', 'Unemployed', 'Retired'], n),
    'education':  rng.choice(['High School', 'Bachelor', 'Master', 'PhD'], n),
    'region':     rng.choice(['North', 'South', 'East', 'West'], n),
    'default':    rng.choice([0, 1], n, p=[0.82, 0.18])
})
for col, pct in [('income', 0.05), ('credit_score', 0.03), ('employment', 0.04)]:
    idx = rng.choice(n, int(n * pct), replace=False)
    df.loc[idx, col] = np.nan

print('Dataset:', df.shape)
print('Missing:')
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
# Define feature sets
numeric_features     = ['age', 'income', 'credit_score', 'loan_amount']
categorical_features = ['employment', 'education', 'region']

X = df[numeric_features + categorical_features]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Build ColumnTransformer
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])
print('ColumnTransformer built.')

In [ ]:
# Full Pipeline with Logistic Regression
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

lr_pipeline.fit(X_train, y_train)
y_pred = lr_pipeline.predict(X_test)
y_prob = lr_pipeline.predict_proba(X_test)[:, 1]

print('Logistic Regression Pipeline:')
print(classification_report(y_test, y_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

In [ ]:
# Full Pipeline with Random Forest
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]
print('Random Forest Pipeline:')
print(classification_report(y_test, rf_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, rf_prob):.4f}')

In [ ]:
# Cross-validation with Pipeline (no leakage across folds!)
cv_scores = cross_val_score(lr_pipeline, X, y, cv=5, scoring='roc_auc', n_jobs=-1)
print('CV ROC-AUC scores:', cv_scores.round(4))
print(f'Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})' )

In [ ]:
# Hyperparameter tuning with GridSearchCV across the full Pipeline
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'classifier__C': [0.01, 0.1, 1.0],
    'classifier__solver': ['lbfgs']
}

grid_search = GridSearchCV(
    lr_pipeline, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=0)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
print('Best CV AUC:', round(grid_search.best_score_, 4))

In [ ]:
# Access fitted steps and feature names
fitted_preprocessor = lr_pipeline.named_steps['preprocessor']
fitted_classifier   = lr_pipeline.named_steps['classifier']

cat_enc = fitted_preprocessor.named_transformers_['cat']['encoder']
cat_feature_names = cat_enc.get_feature_names_out(categorical_features).tolist()
all_feature_names = numeric_features + cat_feature_names
print('Feature names after preprocessing:')
for f in all_feature_names:
    print(' ', f)

In [ ]:
# Save and reload pipeline
import joblib, tempfile

with tempfile.NamedTemporaryFile(suffix='.joblib', delete=False) as f:
    model_path = f.name

joblib.dump(lr_pipeline, model_path)
loaded = joblib.load(model_path)
print('Saved & reloaded match:', (loaded.predict(X_test) == y_pred).all())
os.unlink(model_path)

import os